# MLflow: von Versuchen zum registrierten Modell
**Internet of Production – KI, Woche 3, Block 5** · etwa 25 min · Zweiergruppen

**Roter Faden:** Der Übertragungstest hat gezeigt, dass die eingefrorene Regel an allen sechs Druckern von 2026 funktioniert. Beim Test haben Sie aber mehrere Grenzen und Modelle probiert und die Zahlen im Kopf oder auf Papier behalten. Nächste Woche fragt jemand: *„Welches Modell läuft im Monitor, wie wurde es getestet, und kommen wir zum alten zurück?“* Dafür brauchen Sie ein Protokoll.

**Ziel dieses Notebooks**
1. jeden Versuch als **Run** (Parameter, Kennzahlen, das Modell selbst) in MLflow protokollieren,
2. die Runs in der **Weboberfläche** vergleichen,
3. das gewählte Modell mit Version und dem Alias `production` **registrieren**, damit der Service des nächsten Notebooks es unter seinem Namen laden kann.

**Theorie in je einem Satz:** *Tracking* hält fest, was probiert wurde. Eine *Modell-Registry* gibt einem Modell einen Namen und nummerierte Versionen. Ein *Alias* ist eine verschiebbare Marke (hier `production`), die entscheidet, welche Version live ist; ein Roll-back ist dann eine Änderung der Marke und nicht des Codes.

In [ ]:
import sys, subprocess, time, logging, warnings
sys.path.insert(0, '..')                                   # blockage_core.py lies in the folder above
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import mlflow, mlflow.pyfunc
logging.getLogger('mlflow').setLevel(logging.ERROR); warnings.filterwarnings('ignore')     # keep the output readable
from blockage_core import *

DATA_OLD = Path('../../00_Data/printfarm')
DATA_NEW = Path('../../00_Data/printfarm_2026')
STORE = Path('..').resolve()
TRACKING_URI = f"sqlite:///{(STORE / 'mlflow.db').as_posix()}"      # one database for the whole class folder (and for the service)
mlflow.set_tracking_uri(TRACKING_URI)
EXPERIMENT = 'printfarm-transfer'
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(STORE / 'mlartifacts').as_uri())
mlflow.set_experiment(EXPERIMENT)

refs = fit_references(DATA_OLD)                             # frozen references of weeks 1 to 3 (printer 1, 2024/25, first 70 %)
printers = {k: make_blockage_data(DATA_NEW, k) for k in range(6)}   # the six printers of 2026
print(f"reference: E120 = {refs['mu']:.3f} +/- {refs['sd']:.3f} Wh/min, z limit {refs['z_limit']:.2f}; {len(printers)} printers loaded; tracking in {TRACKING_URI}")

## Schritt 1: Runs protokollieren
`log_run` bewertet ein Modell an allen sechs Druckern von 2026 (das Modell bleibt eingefroren) und schreibt drei Dinge nach MLflow: die **Parameter**, die den Versuch festlegen, die **Kennzahlen** und das **Modell** als Artefakt.

Kennzahlen: `auc` (Mittel über die Drucker), `announced_share` (Blockaden mit mindestens einem Alarm in den 6 Stunden davor), `false_episodes_per_week` (Mittel je Drucker). Die drei Kennzahlen bleiben absichtlich getrennt: Eine einzelne Zahl würde den Zielkonflikt zwischen verpassten Blockaden und Fehlalarmen verdecken.

Lassen Sie die z-Wert-Regel mit drei Grenzen und das KNN mit zwei Entscheidungsschwellen laufen. Wählen Sie eine **niedrigere** und eine **höhere** z-Grenze um die Vorgabe.

In [ ]:
def log_run(name, model, params):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        metrics = evaluate_on_farm(printers, model)
        mlflow.log_metrics(metrics)
        mlflow.pyfunc.log_model(name='model', python_model=model, input_example=refs['ref'][FEATURES].head(2).astype(float),
                                pip_requirements=['mlflow', 'scikit-learn', 'pandas', 'numpy'])
    return metrics

for z_limit in [1.5, round(refs['z_limit'], 2), 2.2]:
    model = ZRuleModel(refs['mu'], refs['sd'], z_limit)
    log_run(f'zrule z>{z_limit}', model, {'model': 'zrule', 'z_limit': z_limit, 'mu': round(refs['mu'], 4), 'sd': round(refs['sd'], 4)})

for threshold in [0.5, 0.3]:
    model = KnnModel(refs['knn'], threshold)
    log_run(f'knn k=25 thr={threshold}', model, {'model': 'knn', 'k': 25, 'threshold': threshold})
print('5 runs logged')

## Schritt 2: Runs vergleichen
Lesen Sie die Runs aus der Datenbank zurück und zeichnen Sie den Zielkonflikt. Jeder Punkt ist ein Versuch; der beste liegt **oben links** (viele Blockaden angekündigt, wenig Fehlalarme).

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=['start_time ASC'])
table = runs[['tags.mlflow.runName', 'metrics.auc', 'metrics.announced_share', 'metrics.false_episodes_per_week']].copy()
table.columns = ['run', 'AUC', 'announced share', 'false episodes / week']
display(table.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(table['false episodes / week'], table['announced share'])
for _, r in table.iterrows():
    ax.annotate(r['run'], (r['false episodes / week'], r['announced share']), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('false alarm episodes per printer and week'); ax.set_ylabel('share of blockages announced (6 h)'); ax.grid(alpha=.3)
plt.show()

**Welchen Run würden Sie in Produktion nehmen? Wovon hängt die Antwort ab? Warum reicht die AUC allein nicht zur Auswahl?**

Die z-Wert-Regel mit der Vorgabe (z = 1,79) ist ein sinnvoller Kompromiss; die niedrigere Grenze kündigt mehr Blockaden an, zum Preis deutlich mehr Fehlalarme, die höhere umgekehrt. Die Wahl hängt vom Kostenverhältnis zwischen Prüfung und Blockade ab und davon, wie viele Fehlalarme das Bedienpersonal toleriert, bevor es nicht mehr reagiert. Die AUC der z-Wert-Regel ist für alle drei Grenzen gleich (sie verschieben nur den Arbeitspunkt auf einer ROC-Kurve) und kann daher nicht zwischen ihnen entscheiden; sie vergleicht nur die Ordnungsgüte verschiedener Modelle.

## Schritt 3: Die Weboberfläche
Starten Sie die MLflow-Oberfläche und öffnen Sie sie. **Lokal / am Poolrechner:** `http://localhost:5000`. **Auf Binder:** den Link anklicken, der unter der Zelle erscheint (er zeigt auf denselben Server hinter dem Binder-Proxy, `…/proxy/5000/`). Der Start dauert einige Sekunden.

Aufgaben im Browser (5 min):
1. Öffnen Sie das Experiment `printfarm-transfer`, wählen Sie zwei Runs und klicken Sie auf **Compare**.
2. Öffnen Sie einen Run: Parameter, Kennzahlen und das Modell-Artefakt suchen (sehen Sie sich `MLmodel` und `requirements.txt` an).
3. Sortieren Sie die Tabelle nach `false_episodes_per_week`.

In [ ]:
import os, requests
from IPython.display import HTML, display
env = dict(os.environ, MLFLOW_SERVER_ENABLE_JOB_EXECUTION='false')    # no background job workers: saves about 2 GB of memory (decisive on Binder, which has 2 GB)
ui = subprocess.Popen([sys.executable, '-m', 'mlflow', 'ui', '--backend-store-uri', TRACKING_URI, '--host', '127.0.0.1', '--port', '5000', '--workers', '1',
                       '--allowed-hosts', '*', '--cors-allowed-origins', '*'],     # Binder forwards its own host name; without this MLflow answers 'Invalid Host header'
                      env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(90):                                                     # wait until the server answers
    try:
        requests.get('http://127.0.0.1:5000/', timeout=1); break
    except requests.RequestException:
        time.sleep(1)
prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '')                # set on Binder / JupyterHub, e.g. /user/<name>/
print('MLflow UI is running. Local: http://localhost:5000')
if prefix:
    display(HTML(f'<a href="{prefix}proxy/5000/" target="_blank">Binder / JupyterHub: open the MLflow UI</a>'))

**Was finden Sie im Run, das Sie nicht selbst protokolliert haben? Welche Angaben bräuchten Sie, um das Ergebnis in einem Jahr zu reproduzieren?**

MLflow ergänzt Startzeit, Dauer, Benutzer, die Quelldatei und die Paketversionen (`requirements.txt`) des Modells. Für die Reproduktion fehlen noch: die genauen Daten (Version oder Hash der Dateien von 2026), der Code-Stand der Merkmalsberechnung (hier `blockage_core.py`) und die Auswertungsregel (6-h-Fenster, Definition einer Fehlalarm-Episode). Das lässt sich als Tags oder Parameter mitschreiben.

## Schritt 4: Modelle registrieren
Durch die Registrierung erhält das Modell einen **Namen** und eine **Versionsnummer**. Der Alias `production` markiert die Version, die der Service lädt. Registrieren Sie die z-Wert-Regel mit der Grenze Ihrer Wahl und das KNN mit Schwelle 0,5.

In [ ]:
CHOSEN_Z_LIMIT = round(refs['z_limit'], 2)

client = mlflow.MlflowClient()
def register(name, model, params):
    with mlflow.start_run(run_name=f'register {name}'):
        mlflow.log_params(params)
        info = mlflow.pyfunc.log_model(name='model', python_model=model, input_example=refs['ref'][FEATURES].head(2).astype(float),
                                       pip_requirements=['mlflow', 'scikit-learn', 'pandas', 'numpy'])
    version = mlflow.register_model(info.model_uri, name)
    client.set_registered_model_alias(name, 'production', version.version)
    print(f'{name}: version {version.version} has the alias production')

register('blockage-zrule', ZRuleModel(refs['mu'], refs['sd'], CHOSEN_Z_LIMIT), {'model': 'zrule', 'z_limit': CHOSEN_Z_LIMIT})
register('blockage-knn', KnnModel(refs['knn'], 0.5), {'model': 'knn', 'k': 25, 'threshold': 0.5})

Prüfen Sie im Browser unter **Models**: Beide Modelle sollten mit Version und Alias erscheinen. Beenden Sie dann die Oberfläche (die Registry bleibt in `mlflow.db`).

In [ ]:
ui.terminate()
for m in client.search_registered_models():
    print(m.name, {a: v for a, v in m.aliases.items()})

**Die Registry speichert das Modell, aber nicht die *Merkmale*. Was kann schiefgehen, wenn Trainings- und Betriebscode E120 unterschiedlich berechnen? Wo wird das in diesem Kurs verhindert?**

Das Modell erhält Zahlen, die richtig aussehen, aber eine andere Bedeutung haben (Training/Serving-Skew): andere Fensterlänge, andere Einheit (kWh statt Wh), andere Behandlung von Leerlaufminuten. Es liefert trotzdem einen Score, und niemand merkt es. Hier importieren beide Seiten dieselben Funktionen aus `blockage_core.py`; Notebook 03 prüft die Gleichheit numerisch. In größeren Systemen übernimmt ein Feature Store oder eine getestete gemeinsame Bibliothek diese Rolle.